In [30]:
from google.colab import drive
import numpy as np
import pandas as pd

drive.mount('/content/drive')

train_path = '/content/drive/MyDrive/train.csv'
test_path = '/content/drive/MyDrive/test.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print('Train shape:', train_df.shape)
print('Test shape :', test_df.shape)
print("\nTarget ('label') distribution in train:")
print(train_df['label'].value_counts(dropna=False))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train shape: (32754, 1240)
Test shape : (8189, 1240)

Target ('label') distribution in train:
label
0    31626
1     1128
Name: count, dtype: int64


In [31]:
y_train = train_df['label'].copy()
y_test = test_df['label'].copy()

X_train = train_df.drop(columns=['label']).copy()
X_test = test_df.drop(columns=['label']).copy()

fp_cols = [c for c in X_train.columns if c.startswith('fp_')]
desc_cols = [c for c in X_train.columns if not c.startswith('fp_')]

print(
    f'Fingerprints count: {len(fp_cols)} | Descriptors count: {len(desc_cols)}'
)
print('First 25 descriptors:', desc_cols[:25])

Fingerprints count: 1024 | Descriptors count: 215
First 25 descriptors: ['MaxAbsEStateIndex', 'MaxEStateIndex', 'MinAbsEStateIndex', 'MinEStateIndex', 'qed', 'SPS', 'MolWt', 'HeavyAtomMolWt', 'ExactMolWt', 'NumValenceElectrons', 'NumRadicalElectrons', 'MaxPartialCharge', 'MinPartialCharge', 'MaxAbsPartialCharge', 'MinAbsPartialCharge', 'FpDensityMorgan1', 'FpDensityMorgan2', 'FpDensityMorgan3', 'BCUT2D_MWHI', 'BCUT2D_MWLOW', 'BCUT2D_CHGHI', 'BCUT2D_CHGLO', 'BCUT2D_LOGPHI', 'BCUT2D_LOGPLOW', 'BCUT2D_MRHI']


In [32]:
interesting = [
    'MolWt',
    'MolLogP',
    'TPSA',
    'NumHDonors',
    'NumHAcceptors',
    'HeavyAtomCount',
    'RingCount',
    'FractionCSP3',
]
present = [c for c in interesting if c in desc_cols]
print('Present interesting descriptors:', present)

Present interesting descriptors: ['MolWt', 'MolLogP', 'TPSA', 'NumHDonors', 'NumHAcceptors', 'HeavyAtomCount', 'RingCount', 'FractionCSP3']


In [33]:
def add_engineered_features(df, fp_columns, desc_columns):
  out = df.copy()

  if len(fp_columns) > 0:
    fp_sum = out[fp_columns].sum(axis=1)
    out['fp_active_count'] = fp_sum
    out['fp_density'] = fp_sum / len(fp_columns)

  eps = 1e-6

  if {'MolWt', 'NumHDonors'}.issubset(desc_columns):
    out['hbd_per_molwt'] = out['NumHDonors'] / (out['MolWt'].abs() + eps)

  if {'MolWt', 'NumHAcceptors'}.issubset(desc_columns):
    out['hba_per_molwt'] = out['NumHAcceptors'] / (out['MolWt'].abs() + eps)

  if {'NumHDonors', 'NumHAcceptors'}.issubset(desc_columns):
    out['hbd_plus_hba'] = out['NumHDonors'] + out['NumHAcceptors']
    out['hbd_to_hba_ratio'] = out['NumHDonors'] / (
        out['NumHAcceptors'].abs() + eps
    )

  if {'TPSA', 'MolWt'}.issubset(desc_columns):
    out['tpsa_molwt_ratio'] = out['TPSA'] / (out['MolWt'].abs() + eps)

  if {'MolLogP', 'MolWt'}.issubset(desc_columns):
    out['logp_molwt_ratio'] = out['MolLogP'] / (out['MolWt'].abs() + eps)

  if {'MolLogP', 'TPSA'}.issubset(desc_columns):
    out['logp_tpsa_product'] = out['MolLogP'] * out['TPSA']

  if {'HeavyAtomCount', 'RingCount'}.issubset(desc_columns):
    out['rings_per_heavy_atom'] = out['RingCount'] / (
        out['HeavyAtomCount'].abs() + eps
    )

  num_cols = out.select_dtypes(include=[np.number]).columns
  out[num_cols] = out[num_cols].replace([np.inf, -np.inf], np.nan)

  return out


X_train_fe = add_engineered_features(X_train, fp_cols, desc_cols)
X_test_fe = add_engineered_features(X_test, fp_cols, desc_cols)

new_cols = [c for c in X_train_fe.columns if c not in X_train.columns]
print(f'Added {len(new_cols)} new features:', new_cols)
print('New shapes -> Train:', X_train_fe.shape, '| Test:', X_test_fe.shape)

Added 10 new features: ['fp_active_count', 'fp_density', 'hbd_per_molwt', 'hba_per_molwt', 'hbd_plus_hba', 'hbd_to_hba_ratio', 'tpsa_molwt_ratio', 'logp_molwt_ratio', 'logp_tpsa_product', 'rings_per_heavy_atom']
New shapes -> Train: (32754, 1249) | Test: (8189, 1249)


In [34]:
target_groups = ['Aromatic', 'Rotatable', 'Fraction', 'Ring', 'Mol']
for grp in target_groups:
  cols = [c for c in desc_cols if grp.lower() in c.lower()]
  print(f'=== {grp} ({len(cols)}) ===')
  print(cols)
  print()

=== Aromatic (3) ===
['NumAromaticCarbocycles', 'NumAromaticHeterocycles', 'NumAromaticRings']

=== Rotatable (1) ===
['NumRotatableBonds']

=== Fraction (1) ===
['FractionCSP3']

=== Ring (4) ===
['NumAliphaticRings', 'NumAromaticRings', 'NumSaturatedRings', 'RingCount']

=== Mol (5) ===
['MolWt', 'HeavyAtomMolWt', 'ExactMolWt', 'MolLogP', 'MolMR']



In [35]:
def add_engineered_features_v2(df, fp_columns, desc_columns):
  out = add_engineered_features(df, fp_columns, desc_columns)
  eps = 1e-6

  if {'NumAromaticRings', 'RingCount'}.issubset(desc_columns):
    out['aromatic_ring_ratio'] = out['NumAromaticRings'] / (
        out['RingCount'].abs() + eps
    )

  if {'NumAromaticHeterocycles', 'NumAromaticRings'}.issubset(desc_columns):
    out['aromatic_hetero_ratio'] = out['NumAromaticHeterocycles'] / (
        out['NumAromaticRings'].abs() + eps
    )

  if {'NumRotatableBonds', 'HeavyAtomCount'}.issubset(desc_columns):
    out['rotatable_per_heavy_atom'] = out['NumRotatableBonds'] / (
        out['HeavyAtomCount'].abs() + eps
    )

  if {'FractionCSP3', 'NumRotatableBonds'}.issubset(desc_columns):
    out['csp3_x_rotatable'] = out['FractionCSP3'] * out['NumRotatableBonds']

  if {'MolMR', 'MolWt'}.issubset(desc_columns):
    out['molmr_per_molwt'] = out['MolMR'] / (out['MolWt'].abs() + eps)

  num_cols = out.select_dtypes(include=[np.number]).columns
  out[num_cols] = out[num_cols].replace([np.inf, -np.inf], np.nan)

  return out


X_train_fe = add_engineered_features_v2(X_train, fp_cols, desc_cols)
X_test_fe = add_engineered_features_v2(X_test, fp_cols, desc_cols)

v2_cols = [c for c in X_train_fe.columns if c not in X_train.columns]
print(f'Total engineered features now: {len(v2_cols)}')
print('V2 new features:', v2_cols[10:])
print('New shapes -> Train:', X_train_fe.shape, '| Test:', X_test_fe.shape)

/tmp/ipykernel_756/3819423917.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out['aromatic_ring_ratio'] = out['NumAromaticRings'] / (
/tmp/ipykernel_756/3819423917.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out['aromatic_hetero_ratio'] = out['NumAromaticHeterocycles'] / (
/tmp/ipykernel_756/3819423917.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) i

Total engineered features now: 15
V2 new features: ['aromatic_ring_ratio', 'aromatic_hetero_ratio', 'rotatable_per_heavy_atom', 'csp3_x_rotatable', 'molmr_per_molwt']
New shapes -> Train: (32754, 1254) | Test: (8189, 1254)


In [36]:
num_cols = X_train_fe.select_dtypes(include=[np.number]).columns

print(
    'NaNs before fill -> Train:',
    X_train_fe[num_cols].isna().sum().sum(),
    '| Test:',
    X_test_fe[num_cols].isna().sum().sum(),
)

medians = X_train_fe[num_cols].median()
X_train_fe[num_cols] = X_train_fe[num_cols].fillna(medians)
X_test_fe[num_cols] = X_test_fe[num_cols].fillna(medians)

print(
    'NaNs after fill  -> Train:',
    X_train_fe[num_cols].isna().sum().sum(),
    '| Test:',
    X_test_fe[num_cols].isna().sum().sum(),
)

NaNs before fill -> Train: 0 | Test: 0
NaNs after fill  -> Train: 0 | Test: 0


In [37]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold

X_base_clean = X_train.copy()
base_num_cols = X_base_clean.select_dtypes(include=[np.number]).columns
X_base_clean[base_num_cols] = X_base_clean[base_num_cols].fillna(
    X_base_clean[base_num_cols].median()
)

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)


def eval_cv(X_data, y_data, name):
  roc_scores, pr_scores = [], []
  for tr_idx, val_idx in skf.split(X_data, y_data):
    model = HistGradientBoostingClassifier(
        max_iter=150, learning_rate=0.06, random_state=42
    )
    model.fit(X_data.iloc[tr_idx], y_data.iloc[tr_idx])
    probs = model.predict_proba(X_data.iloc[val_idx])[:, 1]
    roc_scores.append(roc_auc_score(y_data.iloc[val_idx], probs))
    pr_scores.append(average_precision_score(y_data.iloc[val_idx], probs))
  print(
      f'{name:18s} | ROC-AUC: {np.mean(roc_scores):.4f} '
      f'| PR-AUC: {np.mean(pr_scores):.4f}'
  )


eval_cv(X_base_clean, y_train, 'Original (1239)')
eval_cv(X_train_fe, y_train, 'Engineered (1254)')

Original (1239)    | ROC-AUC: 0.8255 | PR-AUC: 0.4597
Engineered (1254)  | ROC-AUC: 0.8298 | PR-AUC: 0.4649


In [38]:
for i, col in enumerate(desc_cols):
  print(f'{i:3d}: {col}')

  0: MaxAbsEStateIndex
  1: MaxEStateIndex
  2: MinAbsEStateIndex
  3: MinEStateIndex
  4: qed
  5: SPS
  6: MolWt
  7: HeavyAtomMolWt
  8: ExactMolWt
  9: NumValenceElectrons
 10: NumRadicalElectrons
 11: MaxPartialCharge
 12: MinPartialCharge
 13: MaxAbsPartialCharge
 14: MinAbsPartialCharge
 15: FpDensityMorgan1
 16: FpDensityMorgan2
 17: FpDensityMorgan3
 18: BCUT2D_MWHI
 19: BCUT2D_MWLOW
 20: BCUT2D_CHGHI
 21: BCUT2D_CHGLO
 22: BCUT2D_LOGPHI
 23: BCUT2D_LOGPLOW
 24: BCUT2D_MRHI
 25: BCUT2D_MRLOW
 26: AvgIpc
 27: BalabanJ
 28: BertzCT
 29: Chi0
 30: Chi0n
 31: Chi0v
 32: Chi1
 33: Chi1n
 34: Chi1v
 35: Chi2n
 36: Chi2v
 37: Chi3n
 38: Chi3v
 39: Chi4n
 40: Chi4v
 41: HallKierAlpha
 42: Ipc
 43: Kappa1
 44: Kappa2
 45: Kappa3
 46: LabuteASA
 47: PEOE_VSA1
 48: PEOE_VSA10
 49: PEOE_VSA11
 50: PEOE_VSA12
 51: PEOE_VSA13
 52: PEOE_VSA14
 53: PEOE_VSA2
 54: PEOE_VSA3
 55: PEOE_VSA4
 56: PEOE_VSA5
 57: PEOE_VSA6
 58: PEOE_VSA7
 59: PEOE_VSA8
 60: PEOE_VSA9
 61: SMR_VSA1
 62: SMR_VSA10
 6

In [39]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold

zero_var_cols = [c for c in X_train_fe.columns if X_train_fe[c].nunique() <= 1]
print(f'Zero-variance columns in train: {len(zero_var_cols)}')

X_tr_clean = X_train_fe.drop(columns=zero_var_cols).copy()
X_te_clean = X_test_fe.drop(columns=zero_var_cols).copy()
print('Cleaned shapes -> Train:', X_tr_clean.shape, '| Test:', X_te_clean.shape)

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)


def eval_model_cfg(X_data, y_data, cfg_name, **model_params):
  roc_scores, pr_scores = [], []
  for tr_idx, val_idx in skf.split(X_data, y_data):
    model = HistGradientBoostingClassifier(random_state=42, **model_params)
    model.fit(X_data.iloc[tr_idx], y_data.iloc[tr_idx])
    probs = model.predict_proba(X_data.iloc[val_idx])[:, 1]
    roc_scores.append(roc_auc_score(y_data.iloc[val_idx], probs))
    pr_scores.append(average_precision_score(y_data.iloc[val_idx], probs))
  print(
      f'{cfg_name:26s} | ROC-AUC: {np.mean(roc_scores):.4f} '
      f'| PR-AUC: {np.mean(pr_scores):.4f}'
  )


eval_model_cfg(
    X_tr_clean,
    y_train,
    'V2 Clean (lr=0.04, iter=250)',
    learning_rate=0.04,
    max_iter=250,
    min_samples_leaf=30,
)
eval_model_cfg(
    X_tr_clean,
    y_train,
    'V2 Clean + balanced',
    learning_rate=0.04,
    max_iter=250,
    min_samples_leaf=30,
    class_weight='balanced',
)

Zero-variance columns in train: 21
Cleaned shapes -> Train: (32754, 1233) | Test: (8189, 1233)
V2 Clean (lr=0.04, iter=250) | ROC-AUC: 0.8261 | PR-AUC: 0.4666
V2 Clean + balanced        | ROC-AUC: 0.8285 | PR-AUC: 0.4025


In [40]:
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()

X_tr_scaled = pd.DataFrame(
    scaler.fit_transform(X_tr_clean),
    columns=X_tr_clean.columns,
    index=X_tr_clean.index,
)
X_te_scaled = pd.DataFrame(
    scaler.transform(X_te_clean),
    columns=X_te_clean.columns,
    index=X_te_clean.index,
)

print("Scaled shapes Train:", X_tr_scaled.shape, "| Test:", X_te_scaled.shape)
print("NaNs after scaling Train:", X_tr_scaled.isna().sum().sum(),
      "| Test:", X_te_scaled.isna().sum().sum())

Scaled shapes Train: (32754, 1233) | Test: (8189, 1233)
NaNs after scaling Train: 0 | Test: 0


In [ ]:
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier  #Sanity check
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

hgb_roc, hgb_pr = [], []
et_roc, et_pr = [], []
blend_roc, blend_pr = [], []

for tr_idx, val_idx in skf.split(X_tr_clean, y_train):
  X_tr, y_tr = X_tr_clean.iloc[tr_idx], y_train.iloc[tr_idx]
  X_va, y_va = X_tr_clean.iloc[val_idx], y_train.iloc[val_idx]

  hgb = HistGradientBoostingClassifier(
      learning_rate=0.04, max_iter=250, min_samples_leaf=30, random_state=42
  )
  et = ExtraTreesClassifier(n_estimators=250, n_jobs=-1, random_state=42)

  hgb.fit(X_tr, y_tr)
  et.fit(X_tr, y_tr)

  p_hgb = hgb.predict_proba(X_va)[:, 1]
  p_et = et.predict_proba(X_va)[:, 1]
  p_blend = 0.7 * p_hgb + 0.3 * p_et

  hgb_roc.append(roc_auc_score(y_va, p_hgb))
  hgb_pr.append(average_precision_score(y_va, p_hgb))

  et_roc.append(roc_auc_score(y_va, p_et))
  et_pr.append(average_precision_score(y_va, p_et))

  blend_roc.append(roc_auc_score(y_va, p_blend))
  blend_pr.append(average_precision_score(y_va, p_blend))

print(
    f'HGB alone   | ROC-AUC: {np.mean(hgb_roc):.4f} | PR-AUC: {np.mean(hgb_pr):.4f}'
)
print(
    f'ET alone    | ROC-AUC: {np.mean(et_roc):.4f} | PR-AUC: {np.mean(et_pr):.4f}'
)
print(
    f'Blend 0.7/0.3 | ROC-AUC: {np.mean(blend_roc):.4f} | PR-AUC:'
    f' {np.mean(blend_pr):.4f}'
)

In [42]:
train_fe_out = X_train_fe.copy()
train_fe_out['label'] = y_train.values

test_fe_out = X_test_fe.copy()
test_fe_out['label'] = y_test.values

train_fe_path = '/content/drive/MyDrive/train_fe.csv'
test_fe_path = '/content/drive/MyDrive/test_fe.csv'

train_fe_out.to_csv(train_fe_path, index=False)
test_fe_out.to_csv(test_fe_path, index=False)

print('Saved successfully', train_fe_out.shape, test_fe_out.shape)

Saved successfully -> (32754, 1255) (8189, 1255)
